[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/developability/developability.ipynb)

# Developable Surfaces

This example flows a curved surface, the Stanford bunny, towards a developable one, following Stein, Grinspun and Crane. A developable surface can be unrolled flat without stretching or tearing, like a sheet of paper bent into a cylinder or a cone; it bends in one direction only at each point. The bunny's rounded flanks bend both ways almost everywhere, and flowing it towards developability creases it into patches that could each be cut from a flat sheet, joined along sharp folds.

The computation is written in the geometric algebra of three-dimensional space. How far a vertex is from developable is the smallest eigenvalue of a map, a sum of dyads of the face normals around it, and the flow's gradient is a wedge of two vectors made a vector by the dual.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack

from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.surfaces.developability import render
from examples.surfaces.developability.scenarios import bunny
from examples.surfaces.developability.core import Mesh, as_scalar, mv

In [ ]:
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()

## 1. The spread of the normals around a vertex

A smooth surface is developable where its normals, all around a point, lie on one great circle of the sphere of directions: the surface bends about one direction only, the direction of its rulings, the straight lines that run through a developable surface, and the normals turn about that direction. At a vertex of a triangle mesh, the normals of the faces around it lie on one great circle exactly when one direction meets every one of them at a right angle. The spread of the normals, the sum of their dyads `normal * (normal | Vector)` over the faces around the vertex, each weighted by the face's angle at the vertex, is a map whose quadratic form measures, for a direction `u`, how much the normals lean along it, `sum(angle * (u | normal)**2)`. Its smallest eigenvalue is zero when some direction is perpendicular to every normal, and grows as the normals spread out over the sphere. The developability energy is the sum of the smallest eigenvalues over the vertices.

The angles make the spread a property of the surface rather than of its triangles: a face split in two keeps its angle at the vertex, so splitting faces changes nothing. Without them, a flow can lower the energy by squeezing faces into slivers instead of creasing the surface.

In the notation of the paper the energy reads $E = \sum_i \lambda_{\min}\left(\sum_{ijk} \theta_i^{jk} N_{ijk} N_{ijk}^{\mathsf T}\right)$, with $\theta_i^{jk}$ the angle of face $ijk$ at vertex $i$.

In [ ]:
def spread(mesh: Mesh) -> tuple[Vector, Vector]:
    """The unit normal of each face, and the spread of the face normals around each vertex, the sum of
    their dyads, each weighted by the face's angle at the vertex, a map `Vector <- Vector` at each vertex."""
    edges = mesh.triangle_edges                                                 # [3] Vector[F]
    normals = (edges[0] ^ edges[1]).dual().normalized()                         # Vector[F]
    # Each face to the vertices at its corners through its angles there, run back: each vertex sums over
    # its faces.
    around = ~mesh.at_corners(mesh.corner_cosines().arccos())                   # [V, F] Scalar
    return normals, around * (normals * (normals | Vector))                    # Vector[F], [V] Vector <- Vector


rabbit = bunny()
normals, spreads = spread(rabbit)
values, vectors = spreads.batch().eigh()                                        # [V, 3] Scalar, [V, 3] Vector
print("the bunny's developability energy:", values[:, 0].sum(axis=0).to_array())

## 2. The gradient

The flow moves every vertex down the energy's gradient. The smallest eigenvalue is the least value of the spread's form over unit directions, so to first order its change is the change of the form at the least direction `u`, held fixed: each face's term, its angle times `(u | normal)**2`, changes as the face's normal turns and as its angle opens.

Moving one corner of a face turns the face's normal by the part of the step across the edge opposite the corner, divided by twice the face's area. For each face, the least directions of its three corners enter only through the sum of their angle-weighted dyads applied to the face's normal; its part along the surface, wedged with the edge opposite a corner and made a vector by the dual, over twice the face's area, is that corner's share of the gradient.

A corner's angle opens as either neighbour moves along the face, square to the edge between them, by the step over that edge's length. Wedging the corner's two edges gives the plane they span, and `(plane | edge) / (edge | edge)`, over the plane's size, is that direction over that length: the inner product of a plane with a vector in it turns the vector a quarter turn within the plane. The corner itself moves against both. Each angle's term weighs these by `(u | normal)**2`. The shares, run back from the faces to their corners, add up at the vertices.

In [ ]:
def gradient(mesh: Mesh, normals: Vector, directions: Vector) -> Vector:
    """The energy's gradient at each vertex, with each vertex's least direction `[V] Vector` held fixed."""
    edges = mesh.triangle_edges.batch()                                         # [3, F] Vector
    doubled = (edges[0] ^ edges[1]).norm()                                      # [F] Scalar: twice the area
    face_normals = normals.batch()                                              # [F] Vector
    angles = mesh.corner_cosines().batch().arccos()                             # [3, F] Scalar
    corner_directions = directions[mesh.faces.T]                                # [3, F] Vector
    along = corner_directions | face_normals                                    # [3, F] Scalar
    # As the normal turns: the angle-weighted dyads of the corners' least directions, applied to the face's
    # normal, their part along the face, wedged with the edge facing each corner, over twice the area.
    pulled = (corner_directions * (along * angles)).sum(axis=0)                 # [F] Vector
    across = 2 * (pulled - face_normals * (face_normals | pulled))             # [F] Vector
    turning = (across ^ edges).dual() / doubled                                 # [3, F] Vector
    # As the angle opens: the edges from each corner to the next and to the one after, and their plane.
    ahead, behind = edges[[2, 0, 1]], -edges[[1, 2, 0]]                         # [3, F] Vector each
    plane = ahead ^ behind                                                      # [3, F] Bivector
    weight = along * along / plane.norm()                                       # [3, F] Scalar
    to_next = (plane | ahead) / (ahead | ahead) * weight                        # [3, F] Vector
    to_after = -(plane | behind) / (behind | behind) * weight                   # [3, F] Vector
    opening = to_next[[2, 0, 1]] + to_after[[1, 2, 0]] - to_next - to_after     # [3, F] Vector
    return ~mesh.at_corners(turning + opening) * as_scalar(np.ones(len(mesh.faces))).field()   # Vector[V]


print("largest gradient on the bunny:", gradient(rabbit, normals, vectors[:, 0]).norm().to_array().max())

## 3. The flow

Each step of the flow computes the spread and its least directions afresh, and moves every vertex a small step down the gradient, divided by how sharply the energy curves at the vertex. Moving a corner turns its face's normal by the step over the corner's height above the opposite edge, so each face adds its angle at the vertex over that height squared. A step of the same size everywhere overshoots where triangles are small or thin, at the tips of the ears first, and the flow jitters there; over the stiffness, every vertex moves by the same share of what its faces can take. Before and after, the bunny is drawn with every face lit flat by its own normal, so a developable patch reads as a smooth sweep of shade and a fold as a sharp change of it. The rounded flanks, back and head settle into a few large patches, joined along creases; along a fold the normals take two directions only, so a fold is developable too.

In [ ]:
def stiffness(mesh: Mesh) -> Scalar:
    """How sharply the energy curves at each vertex: each face's angle at the vertex over the vertex's
    height above the opposite edge squared, `(edge | edge) / doubled**2`."""
    edges = mesh.triangle_edges.batch()                                         # [3, F] Vector
    doubled = (edges[0] ^ edges[1]).norm()                                      # [F] Scalar
    angles = mesh.corner_cosines().batch().arccos()                             # [3, F] Scalar
    return ~mesh.at_corners(angles * (edges | edges) / (doubled * doubled)) * as_scalar(np.ones(len(mesh.faces))).field()


def flow(mesh: Mesh, step: float, steps: int, every: int):
    """Gradient descent on the developability energy, each vertex's step the gradient over the stiffness:
    every `every` steps, and at the end, the vertices and each vertex's smallest eigenvalue."""
    vertices = mesh.vertices
    for count in range(steps + 1):
        current = mesh.copy(vertices)
        normals, spreads = spread(current)
        values, vectors = spreads.batch().eigh()                                # [V, 3] Scalar, [V, 3] Vector
        if count % every == 0 or count == steps:
            yield vertices, values[:, 0]
        vertices = vertices - step * gradient(current, normals, vectors[:, 0]) / stiffness(current)


# The flow's step, as a share of what each vertex's faces can take, its number of steps, and the steps
# between kept states.
step, steps, every = 0.3, 3000, 50
states = list(flow(rabbit, step, steps, every))
energies = stack([values.sum(axis=0) for _, values in states])                  # [states] Scalar
print("energy before and after:")
print(energies[0].coefficient_table())
print(energies[-1].coefficient_table())
save_figure(render.draw_before_after([(rabbit, states)]), "developability");

## 4. The bunny creasing

The flow, every fiftieth step: the bumps of the scan smooth out first, then the flanks flatten into patches and the folds between them sharpen.

In [ ]:
show_animation(render.animate_flow(rabbit, states), "developability_flow", 80)

In [ ]:
# checks
# Every kept state has less energy than the one before, and the flow takes away nearly all of it.
assert np.all(np.diff(energies) < 0)
assert energies[-1] < 0.1 * energies[0]